In [ ]:

import csv

import os

import os

from collections import Counter, defaultdict

from datetime import datetime

from scapy.all import rdpcap, IP, TCP, UDP, ICMP, DNS

Set project configurations


In [ ]:
import os


PCAP_FILE = "/Users/yasin/Desktop/project/Network-Traffic-Analysis/captures/network_traffic.pcapng"


REPORT_DIR = "reports"


PORT_SCAN_THRESHOLD = 10


if not os.path.exists(PCAP_FILE):
    if not os.path.isabs(PCAP_FILE):
        PCAP_FILE = os.path.abspath(PCAP_FILE)

    if not os.path.exists(PCAP_FILE):
        raise FileNotFoundError(
            f"PCAP file not found: {PCAP_FILE}. "
            "Place the capture at captures/network_traffic.pcap."
        )

    print(f"Using PCAP file: {PCAP_FILE}")


Read the PCAP file

In [ ]:

try:

    packets = rdpcap(PCAP_FILE)
    print("Total captured packets:", len(packets))

except Exception as error:

    print("Unable to read PCAP file:", error)

    exit()

Total captured packets: 1421


Create counters and storage variables

In [ ]:

protocol_counts = Counter()

source_counts = Counter()

destination_counts = Counter()

dns_queries = Counter()

syn_ports_by_pair = defaultdict(set)

packet_rows = []

ipv4_count = 0

Analyze network packets

In [ ]:

for packet in packets:

    if not packet.haslayer(IP):

        continue

    ip_packet = packet[IP]

    source_ip = ip_packet.src
    destination_ip = ip_packet.dst

    ipv4_count += 1

    source_port = ""
    destination_port = ""

    if packet.haslayer(TCP):

        protocol = "TCP"

        source_port = packet[TCP].sport
        destination_port = packet[TCP].dport

        tcp_flags = int(packet[TCP].flags)

        is_initial_syn = (tcp_flags & 0x02) and not (tcp_flags & 0x10)

        if is_initial_syn:

            syn_ports_by_pair[(source_ip, destination_ip)].add(destination_port)

    elif packet.haslayer(UDP):

        protocol = "UDP"

        source_port = packet[UDP].sport
        destination_port = packet[UDP].dport

    elif packet.haslayer(ICMP):

        protocol = "ICMP"

    else:

        protocol = "OTHER"

    protocol_counts[protocol] += 1

    source_counts[source_ip] += 1

    destination_counts[destination_ip] += 1

    if packet.haslayer(DNS):

        dns_packet = packet[DNS]

        if dns_packet.qr == 0 and dns_packet.qd is not None:

            query_name = dns_packet.qd.qname

            if isinstance(query_name, bytes):

                query_name = query_name.decode("utf-8", errors="ignore")

            dns_queries[query_name.rstrip(".")] += 1

    timestamp = datetime.fromtimestamp(
        float(packet.time)
    ).strftime("%Y-%m-%d %H:%M:%S")

    packet_rows.append({

        "Timestamp": timestamp,

        "Source IP": source_ip,

        "Destination IP": destination_ip,

        "Protocol": protocol,

        "Source Port": source_port,

        "Destination Port": destination_port,

        "Packet Length": len(packet)

    })

Detect possible suspicious activit

In [ ]:

security_alerts = []

for ip_pair, destination_ports in syn_ports_by_pair.items():

    source_ip, destination_ip = ip_pair

    unique_port_count = len(destination_ports)

    if unique_port_count >= PORT_SCAN_THRESHOLD:

        alert = (
            f"Possible TCP SYN port-scan pattern: "
            f"{source_ip} sent SYN packets to "
            f"{unique_port_count} unique ports on {destination_ip}"
        )

        security_alerts.append(alert)

Generate packet-level CSV report

In [ ]:

os.makedirs(REPORT_DIR, exist_ok=True)

csv_file = os.path.join(REPORT_DIR, "packet_analysis.csv")

fieldnames = [
    "Timestamp",
    "Source IP",
    "Destination IP",
    "Protocol",
    "Source Port",
    "Destination Port",
    "Packet Length"
]

with open(csv_file, "w", newline="", encoding="utf-8") as file:

    writer = csv.DictWriter(file, fieldnames=fieldnames)

    writer.writeheader()

    writer.writerows(packet_rows)

print("Packet CSV report generated:", csv_file)

Packet CSV report generated: reports/packet_analysis.csv


Generate the summary report

In [ ]:

report_lines = []

report_lines.append("NETWORK TRAFFIC ANALYSIS REPORT")

report_lines.append("=" * 40)

report_lines.append(f"Total Captured Packets: {len(packets)}")

report_lines.append(f"IPv4 Packets Analyzed: {ipv4_count}")

report_lines.append("\nPROTOCOL ANALYSIS")

report_lines.append("-" * 40)

for protocol, count in protocol_counts.most_common():

    report_lines.append(f"{protocol}: {count}")


report_lines.append("\nTOP SOURCE IP ADDRESSES")

report_lines.append("-" * 40)

for ip, count in source_counts.most_common(5):

    report_lines.append(f"{ip}: {count} packets")


report_lines.append("\nTOP DESTINATION IP ADDRESSES")

report_lines.append("-" * 40)

for ip, count in destination_counts.most_common(5):

    report_lines.append(f"{ip}: {count} packets")


report_lines.append("\nDNS QUERY ANALYSIS")

report_lines.append("-" * 40)

for domain, count in dns_queries.most_common(10):

    report_lines.append(f"{domain}: {count} queries")


report_lines.append("\nSECURITY ALERTS")

report_lines.append("-" * 40)

if security_alerts:

    for alert in security_alerts:

        report_lines.append(alert)

else:

    report_lines.append("No possible port-scan patterns detected.")


report_file = os.path.join(REPORT_DIR, "traffic_summary.txt")

with open(report_file, "w", encoding="utf-8") as file:

    file.write("\n".join(report_lines))


print("\nAnalysis completed successfully!")

print("Summary report generated:", report_file)


Analysis completed successfully!
Summary report generated: reports/traffic_summary.txt
